# baseline regression

In [3]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

# ==========================================================
# 1. Read data and perform basic cleaning
# ==========================================================
base = r"C:\Users\SC2zh\Desktop\S4 paper\DDDdata"
df = pd.read_csv(fr"{base}\14100355.csv")

df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.dropna(subset=["date"])

df["province"] = df["province"].astype(str).str.strip()
df["industry"] = df["industry"].astype(str).str.strip()

df = df.dropna(subset=["employment_rate"]).copy()
df["min_wage"] = pd.to_numeric(df["min_wage"], errors="coerce")

df = df.sort_values(["province", "date"])

# ==========================================================
# 2. Identify minimum wage increase events occurring in 2023+
# ==========================================================
df["mw_change"] = df.groupby("province")["min_wage"].diff()

# Indicator for minimum wage increases occurring in 2023 or later
df["mw_up_2023"] = (
    (df["date"] >= "2023-01-01") &
    (df["mw_change"] > 0)
).astype(int)

# ==========================================================
# 3. Once-treated-always-treated post indicator (2023 definition)
#    A province is considered post-treatment once it experiences
#    at least one minimum wage increase after 2023
# ==========================================================
df["post"] = df.groupby("province")["mw_up_2023"].transform("cumsum")
df["post"] = (df["post"] > 0).astype(int)

# Treated indicator: whether a province ever experienced
# a minimum wage increase in 2023 or later
df["treated"] = df.groupby("province")["mw_up_2023"].transform("max")

print("=== First entry into post-treatment period by province (2023+ events only) ===")
print(df.loc[df["post"] == 1].groupby("province")["date"].min())
print()

# ==========================================================
# 4. Industry-level simple DID regressions:
#    employment_rate ~ treated × post + fixed effects
# ==========================================================

results = []
industries = sorted(df["industry"].unique())

for ind in industries:

    df_ind = df[df["industry"] == ind].copy()

    # Skip industries where treated or post lacks variation
    if (df_ind["treated"].nunique() < 2) or (df_ind["post"].nunique() < 2):
        continue

    clusters = df_ind["province"].astype("category").cat.codes

    df_ind["did"] = df_ind["treated"] * df_ind["post"]

    formula = """
    employment_rate ~ did + C(province) + C(date)
    """

    try:
        model = smf.ols(formula, data=df_ind).fit(
            cov_type="cluster",
            cov_kwds={"groups": clusters}
        )

        coef = model.params.get("did", np.nan)
        se = model.bse.get("did", np.nan)
        pval = model.pvalues.get("did", np.nan)

        ci_low = coef - 1.96 * se if pd.notnull(se) else np.nan
        ci_high = coef + 1.96 * se if pd.notnull(se) else np.nan

        results.append({
            "industry": ind,
            "coef_did": coef,
            "se": se,
            "p_value": pval,
            "ci_low": ci_low,
            "ci_high": ci_high,
            "n_obs": len(df_ind)
        })

    except Exception as e:
        print(f"Error in {ind}: {e}")

# ==========================================================
# 5. Output results
# ==========================================================
res_df = pd.DataFrame(results)

print("\n=== Simple DID (minimum wage increases in 2023+) ===")
print(
    res_df.sort_values("p_value")
          .to_string(index=False, float_format=lambda x: f"{x: .6f}")
)

print("\n=== Statistically significant industries (p < 0.10) ===")
print(
    res_df[res_df["p_value"] < 0.10]
          .to_string(index=False, float_format=lambda x: f"{x: .6f}")
)


=== First entry into post-treatment period by province (2023+ events only) ===
province
British Columbia            2023-06-01
Manitoba                    2023-04-01
New Brunswick               2023-04-01
Newfoundland and Labrador   2023-04-01
Nova Scotia                 2023-04-01
Ontario                     2024-10-01
Prince Edward Island        2023-10-01
Quebec                      2023-05-01
Saskatchewan                2024-10-01
Name: date, dtype: datetime64[ns]


=== Simple DID (minimum wage increases in 2023+) ===
                                           industry  coef_did        se   p_value    ci_low   ci_high  n_obs
    Professional, scientific and technical services -0.023657  0.013490  0.079486 -0.050097  0.002783    330
                     Transportation and warehousing -0.017198  0.011330  0.129015 -0.039405  0.005008    330
                Information, culture and recreation -0.012452  0.008466  0.141330 -0.029045  0.004141    330
                                    